In [45]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings("ignore")

#### Step 1: Create an imbalanced binary classification dataset

In [46]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=2,
    n_redundant=8,
    weights=[0.9, 0.1],
    flip_y=0,
    random_state=42
)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

#### Split the dataset into training and testing sets

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

#### Experiment 1: Train Logistic Regression Classifier

In [48]:
log_reg = LogisticRegression(
    C=1,
    solver="liblinear"
)

log_reg.fit(X_train, y_train)

y_pred_log_reg = log_reg.predict(X_test)

print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



#### Experiment 2: Train Random Forest Classifier

In [49]:
rf_clf = RandomForestClassifier(
    n_estimators=30,
    max_depth=3
)

rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



#### Experiment 3: Train XGBoost

In [50]:
xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_clf.fit(X_train, y_train)

y_pred_xgb = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [51]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)

X_train_res, y_train_res = smt.fit_resample(
    X_train,
    y_train
)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [52]:
xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_clf.fit(X_train_res, y_train_res)

y_pred_xgb = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98       270
           1       0.78      0.83      0.81        30

    accuracy                           0.96       300
   macro avg       0.88      0.90      0.89       300
weighted avg       0.96      0.96      0.96       300



#### Track Experiments Using MLFlow

In [53]:
models = [
    (
        "Logistic Regression",
        {"C": 1, "solver": "liblinear"},
        LogisticRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": "logloss"},
        XGBClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": "logloss"},
        XGBClassifier(),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [54]:
reports = []

for model_name, params,model, train_set, test_set in models:

    X_train = train_set[0]
    y_train = train_set[1]

    X_test = test_set[0]
    y_test = test_set[1]
    model.set_params(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    reports.append(report)

In [55]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9607142857142857,
   'recall': 0.9962962962962963,
   'f1-score': 0.9781818181818182,
   'support': 270.0},
  '1': {'precision': 0.95,
   'recall': 0.6333333333333333,
   'f1-score': 0.76,
   'support': 30.0},
  'accuracy': 0.96,
  'macro avg': {'precision': 0.9553571428571428,
   'recall': 0.8148148148148149,
   'f1-score': 0.8690909090909091,
   'support': 300.0},
  'weighted avg': {'precision': 0.9596428571428572,
   'recall':

In [56]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Dagshub setup

In [59]:
# !pip install dagshub

In [60]:
import dagshub
dagshub.init(repo_owner='ankitgupta1729', repo_name='mlflow_dagshub_demo', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6852d2d1-02b9-40bb-b6cf-1e1cb3f0cdca&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a1afb6de444d96915c69946ddc5071cea644f4a37047cb8a6e870a783705fef8




Accessing as ankitgupta1729

Initialized MLflow to track repo "ankitgupta1729/mlflow_dagshub_demo"

Repository ankitgupta1729/mlflow_dagshub_demo initialized!

In [61]:
#mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")
mlflow.set_tracking_uri(uri="https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow")
mlflow.set_experiment("Anomaly Detection Model Registry")

for i, element in enumerate(models):
    model_name, params, model, train_set, test_set = element
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("accuracy", report["accuracy"])
        mlflow.log_metric("recall_class_1", report["1"]["recall"])
        mlflow.log_metric("recall_class_0", report["0"]["recall"])
        mlflow.log_metric("f1_score_macro", report["macro avg"]["f1-score"])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")


2026/07/04 18:57:38 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection Model Registry' does not exist. Creating a new experiment.


🏃 View run Logistic Regression at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/9075d750e0ae4dd999d4eb6237395b1c
🧪 View experiment at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run Random Forest at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/4935239dcf354ca8937de90739b012f4
🧪 View experiment at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run XGBClassifier at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/77e087dab7224e11830d1a2637c5ccf6
🧪 View experiment at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/c053cb1b6f53406686c29255de1fc57f
🧪 View experiment at: https://dagshub.com/ankitgupta1729/mlflow_dagshub_demo.mlflow/#/experiments/0


#### Register the Model

In [41]:
registered_model_name = "XGB-Smote"
logged_model_name = "model"
run_id = input("Enter the run_id of the model you want to register: ")
model_uri = f"runs:/{run_id}/{logged_model_name}"

result = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name,
)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/07/04 18:05:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 119f3b9fb1444cdf9c4ea07e1c04d8ef has no artifacts at artifact path 'model', registering model based on models:/m-135af249d24940faac705da08bc60ae2 instead
2026/07/04 18:05:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


#### Load the model

In [42]:
model_version = 1
model_uri = f"models:/{registered_model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri=model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

##### Alternate way without remembering the model version

In [43]:
model_version = 1
model_uri = f"models:/{registered_model_name}@challenger"

loaded_model = mlflow.xgboost.load_model(model_uri=model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [44]:
dev_model_uri = f"models:/{registered_model_name}@challenger"
prod_model = 'anomaly-detection-prod'

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1783169571976, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1783169571976, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='119f3b9fb1444cdf9c4ea07e1c04d8ef', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>